### Importing necessary packages and files

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import paretoset
import periodictable
from pymatgen.core import Structure
import ast
from sklearn.linear_model import ElasticNet

In [2]:
pdata = pd.read_excel("MATSCI_176_Project_Data.xlsx")
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,structure,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L
0,Li0-3Ag,Li,['Ag'],1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,"{'@module': 'pymatgen.core.structure', '@class...",0.333634,0,417.933696,4.179337
1,Li0-3Sb,Li,['Sb'],1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.744135,0.7074,1920.448911,19.204489
2,Li0-1Bi,Li,['Bi'],1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.034716,0,735.871648,7.358716
3,Li0-3Ce,Li,['Ce'],1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.330999,0,576.149771,5.761498
4,Li0-3Ca,Li,['Ca'],1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.415156,0,82.574983,0.825750


In [3]:
structures = pdata["structure"]
print(structures.shape)
structure_dict_test = ast.literal_eval(structures[1])
structure_test = Structure.from_dict(structure_dict_test)
print(structure_test)
print(structure_test.lattice.abc)
print(structure_test.volume)

(5786,)
Full Formula (Li3 Sb1)
Reduced Formula: Li3Sb
abc   :   4.627959   4.627959   4.627959
angles:  60.000007  60.000010  60.000007
pbc   :       True       True       True
Sites (4)
  #  SP       a      b     c    magmom
---  ----  ----  -----  ----  --------
  0  Li    0.5    0.5   0.5          0
  1  Li    0.25   0.25  0.25        -0
  2  Li    0.75   0.75  0.75        -0
  3  Sb    0     -0     0            0
(4.627959209506349, 4.627958733729667, 4.62795882)
70.08959778250679


In [4]:
def get_volume(structure_str):
    try:
        structure_dict = ast.literal_eval(structure_str)
        structure = Structure.from_dict(structure_dict)
        return structure.volume
    except:
        return np.nan
structure_volums = pdata["structure"].apply(get_volume) # Volumes in Angstrom^3
def get_abc(structure_str):
    try:
        structure_dict = ast.literal_eval(structure_str)
        structure = Structure.from_dict(structure_dict)
        return structure.lattice.abc
    except:
        return np.nan
# structure_abc = pdata["structure"].apply(get_abc) # abc values in Angstrom
# structure_abc_df = pd.DataFrame(structure_abc.tolist(), columns=['a', 'b', 'c'])
pdata = pdata.drop(columns=["structure"], axis=1)

In [5]:
structure_volums = structure_volums.to_frame()
structure_volums = structure_volums.fillna(0)
structure_volums = structure_volums["structure"]
pdata["structure_volume"] = structure_volums.values

In [6]:
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L,structure_volume
0,Li0-3Ag,Li,['Ag'],1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,0.333634,0,417.933696,4.179337,103.086779
1,Li0-3Sb,Li,['Sb'],1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,-0.744135,0.7074,1920.448911,19.204489,70.089598
2,Li0-1Bi,Li,['Bi'],1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,-0.034716,0,735.871648,7.358716,58.627385
3,Li0-3Ce,Li,['Ce'],1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,-0.330999,0,576.149771,5.761498,1918.735425
4,Li0-3Ca,Li,['Ca'],1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,-0.415156,0,82.574983,0.825750,47.631359


In [7]:
numeric_cols = [
    "material_density",
    "formation_energy_per_atom",
    "band_gap"
]

for col in numeric_cols:
    pdata[col] = pd.to_numeric(pdata[col], errors="coerce")
pdata = pdata.dropna()
pdata = pdata.reset_index(drop=True)
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L,structure_volume
0,Li0-3Ag,Li,['Ag'],1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,0.333634,0.0000,417.933696,4.179337,103.086779
1,Li0-3Sb,Li,['Sb'],1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,-0.744135,0.7074,1920.448911,19.204489,70.089598
2,Li0-1Bi,Li,['Bi'],1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,-0.034716,0.0000,735.871648,7.358716,58.627385
3,Li0-3Ce,Li,['Ce'],1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,-0.330999,0.0000,576.149771,5.761498,1918.735425
4,Li0-3Ca,Li,['Ca'],1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,-0.415156,0.0000,82.574983,0.825750,47.631359


In [8]:
pdata_norm = pdata.iloc[:, 6:]
print(pdata_norm.columns)

Index(['max_delta_volume', 'average_voltage', 'capacity_grav', 'capacity_vol',
       'energy_grav', 'energy_vol', 'stability_charge', 'stability_discharge',
       'electrode_density', 'material_density', 'working_ion_cost_per_kWh',
       'formation_energy_per_atom', 'band_gap', 'energy_density',
       'battery_cost_per_L', 'structure_volume'],
      dtype='object')


#### Defining our scoring function(s) to analyze and their input datasets. 
We don't want to train our model on the features which are used for scoring. Therefore, we need to define a new training dataset excluding these features. Our scoring model uses gravimetric energy, gravimetric capacity, battery cost per liter, energy density, and battery voltage, so we removed these from our training data set. Additionally, features like energy volume, capacity volume, and electrode density are correlated with battery cost and energy density (as they're used in the calculations for those features), so we removed those columns from our training data to avoid any overlap. 

In [9]:
# Getting the energy grav, capacity grav, battery cost per L, energy density, and average voltage columns for scoring and removing those and additional columns from the training data
energy_grav = pdata_norm["energy_grav"]
cap_grav = pdata_norm["capacity_grav"]
battery_cost = pdata_norm["battery_cost_per_L"]
energy_density = pdata_norm["energy_density"]
voltage = pdata_norm['average_voltage']
training_data = pdata_norm.drop(columns=["energy_grav", "capacity_grav", "battery_cost_per_L", "energy_density", "average_voltage", "energy_vol", "capacity_vol", "electrode_density"])

# Creating a new training dataset without cost to benchmark our scoring with and without cost, since our cost is not from the same source (Materials Project)
training_data_nocost = training_data.drop(columns=["working_ion_cost_per_kWh"])

In [10]:
def scoring_nocost(e_g,c_g,e_d,v):
    e_g = e_g.to_numpy().astype(float)
    c_g = c_g.to_numpy().astype(float)
    e_d = e_d.to_numpy().astype(float)
    v = v.to_numpy().astype(float)
    score = np.abs(e_d)/np.max(e_d) + np.abs(c_g*e_g)/(np.max(c_g)*np.max(e_g)) + np.abs(v)/np.max(v)
    # score = abs(e_g) + abs(c_g) + abs(e_d) + abs(v)
    return pd.Series(score)

def scoring_withcost(e_g,c_g,b_c,e_d,v):
    e_g = e_g.to_numpy().astype(float)
    c_g = c_g.to_numpy().astype(float)
    b_c = b_c.to_numpy().astype(float)
    e_d = e_d.to_numpy().astype(float)
    v = v.to_numpy().astype(float)
    score = np.abs(e_d)/np.max(e_d) + np.abs(c_g*e_g/b_c)/(np.max(c_g)*np.max(e_g)/np.max(b_c)) + np.abs(v)/np.max(v)
    # score = abs(e_g) + abs(c_g) - abs(b_c) + abs(e_d) + abs(v)
    return pd.Series(score)

### Benchmarking the Ridge & Elastic Net Regressional Models
We chose ridge regression and elasticnet regression to both identify important features (from the ridge component) while zeroing out features which don't affect battery performance (features that don't affect our score.)

In [11]:
# Testing the no-cst scoring function and defining our fit & training datasets.
battery_score = scoring_nocost(energy_grav, cap_grav, energy_density, voltage)

fit_train, fit_test, score_train, score_test = train_test_split(
    training_data_nocost,
    battery_score,
    test_size=0.1,
    random_state=42
)

# Normalization of the fit and training datasets for better performance of the elastic net model, since it is sensitive to the scale of the features
norm_fit_train = pd.DataFrame(
    StandardScaler().fit_transform(fit_train),
    columns=fit_train.columns
)
norm_fit_test = pd.DataFrame(
    StandardScaler().fit_transform(fit_test),
    columns=fit_test.columns
)

In [12]:
ridge = linear_model.Ridge(alpha=1.0).fit(norm_fit_train, score_train, sample_weight=None)
score_pred = ridge.score(norm_fit_test, score_test)
print(f"Effective $R^2$ score: {score_pred:.3f}")

Effective $R^2$ score: 0.275


In [13]:
best = np.where(battery_score == np.max(battery_score))[0][0] 
# print(pdata_norm.iloc[best])
print(pdata.iloc[best])
# print(len(pdata["capacity_grav"]))
# print(len(pdata_norm["capacity_grav"]))
print("Battery Score:", np.max(battery_score))
# print(energy_grav.iloc[best], cap_grav.iloc[best], energy_density.iloc[best], voltage.iloc[best])

battery_formula                Li1-3Ti2(PO4)3
working_ion                                Li
elements                     ['Ti', 'P', 'O']
nelements                                 3.0
formula_charge                    LiTi2(PO4)3
formula_discharge                Li3Ti2(PO4)3
max_delta_volume                     0.086321
average_voltage                     33.065771
capacity_grav                      133.516371
capacity_vol                       421.348595
energy_grav                       4414.821755
energy_vol                       13932.216223
stability_charge                     3.516172
stability_discharge                  0.074915
electrode_density                    3.155782
material_density                     5.482777
working_ion_cost_per_kWh                 10.0
formation_energy_per_atom           -2.067682
band_gap                                  0.0
energy_density                   13932.216223
battery_cost_per_L                 139.322162
structure_volume                  

Looks pretty decent! We expected a lithium battery to be the best performing battery due to stability.

## Performing K-folds cross validation on our ridge regressional model. 
Choosing k-folds is important to determine if our train-validation split was truely indicative of the full system. 

In [14]:
X = training_data
y = battery_score
y = y.fillna(0)
y = y.astype(float)

In [15]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", linear_model.Ridge(alpha=3.0)) # We chose alpha = 3.0 to weight against sparsity in our features.
])

In [16]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
MAE = []
MSE = []
bestmaterial = []

for train_idx, test_idx in kf.split(X):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    #Find best material in predicted set
    bestmaterial.append(X_test.iloc[np.where(y_pred == np.max(y_pred))[0][0]])

    r2_scores.append(r2_score(y_test, y_pred))
    MSE.append(mean_squared_error(y_test, y_pred))
    MAE.append(mean_absolute_error(y_test, y_pred))

print("Mean R2:", np.mean(r2_scores))
print("Mean MSE:", np.mean(MSE))
print("Mean MAE:", np.mean(MAE))
bestmaterial_df = pd.DataFrame(bestmaterial)
bestmaterial_df

Mean R2: 0.34993048799708915
Mean MSE: 0.010115793402641435
Mean MAE: 0.07527138527512567


,max_delta_volume,stability_charge,stability_discharge,material_density,working_ion_cost_per_kWh,formation_energy_per_atom,band_gap,structure_volume
233,2.670845,5.128907,0.764373,1.756927,1.8,-0.032036,0.0,213.587997
3204,0.086321,3.516172,0.074915,5.482777,10.0,-2.067682,0.0,175.556161
662,12.458089,2.756673,0.684408,5.510727,1.2,-1.952354,0.0,428.950481
215,0.043613,5.210484,0.092931,2.503420,1.8,0.047737,0.0,205.792036
4014,1.020934,1.553271,0.027568,3.455668,10.0,-0.151988,0.0,72.366280


In [17]:
# Converting the best material features back to the original scale for comparison with the original dataset, since we normalized the training & test data. 
# To do so, we took the best material from each fold and compared their features to the original datasets to find the approximated battery  (the battery formula with the smallest minimum difference between predicted best and the rest of the dataset). 
for i in range(len(bestmaterial)):
    best_material = bestmaterial_df.iloc[i]
    min_diff = float('inf')
    best_match_index = -1
    for j in range(len(pdata)):
        diff = np.sum(np.abs(best_material - pdata.iloc[j, 6:6+len(best_material)]))
        if diff < min_diff:
            min_diff = diff
            best_match_index = j
    print(f"Best material in fold {i+1}:")
    print(pdata.iloc[best_match_index, 0])

Best material in fold 1:
Ca0.25-0.5C
Best material in fold 2:
Li1-3Ti2(PO4)3
Best material in fold 3:
Mg0-3C
Best material in fold 4:
Ca0-3N2
Best material in fold 5:
Li0-3ClO


## Testing Elastic Net Regression in the same way as Ridge Regression

In [18]:
alpha = 3.0
l1_ratio = 0.5  # 0=ridge-like, 1=lasso-like

enet_pipeline = Pipeline([
    ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000, random_state=42))
])

score_pred_enet = enet_pipeline.fit(fit_train, score_train).score(fit_test, score_test)
print("ElasticNet test R^2:", score_pred_enet)

ElasticNet test R^2: 0.0018712285869602052


In [19]:
#replace Ridge KFold with Elastic Net KFold

X = training_data
y = battery_score.fillna(0).astype(float)

# Elastic Net hyperparameters
alpha = 1.0
l1_ratio = 0.5

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000, random_state=42))
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
MAE = []
MSE = []
bestmaterial = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # store best predicted sample in this fold (same idea as your ridge code)
    bestmaterial.append(X_test.iloc[np.argmax(y_pred)])

    r2_scores.append(r2_score(y_test, y_pred))
    MSE.append(mean_squared_error(y_test, y_pred))
    MAE.append(mean_absolute_error(y_test, y_pred))

print("Mean R^2:", np.mean(r2_scores))
print("Mean MSE:", np.mean(MSE))
print("Mean MAE:", np.mean(MAE))

bestmaterial_df = pd.DataFrame(bestmaterial)
bestmaterial_df

Mean R^2: -0.0038258746746345994
Mean MSE: 0.015779127722164013
Mean MAE: 0.0966471833528552


,max_delta_volume,stability_charge,stability_discharge,material_density,working_ion_cost_per_kWh,formation_energy_per_atom,band_gap,structure_volume
8,1.900682,0.290332,0.002960,0.946023,10.0,0.046672,0.0000,106.898527
19,0.034131,0.036010,0.020862,4.829614,10.0,-1.304004,0.0000,31.265978
0,2.736089,0.003609,0.000000,2.592432,10.0,0.333634,0.0000,103.086779
1,1.569237,0.328351,0.000000,3.378028,10.0,-0.744135,0.7074,70.089598
3,2.951183,0.000000,0.333634,2.583328,10.0,-0.330999,0.0000,1918.735425


In [20]:
for i in range(len(bestmaterial)):
    best_material = bestmaterial_df.iloc[i]
    min_diff = float('inf')
    best_match_index = -1
    for j in range(len(pdata)):
        diff = np.sum(np.abs(best_material - pdata.iloc[j, 6:6+len(best_material)]))
        if diff < min_diff:
            min_diff = diff
            best_match_index = j
    print(f"Best material in fold {i+1}:")
    print(pdata.iloc[best_match_index, 0])

Best material in fold 1:
Li0-3Sb
Best material in fold 2:
Li0-1Ca2Pb
Best material in fold 3:
Li0-3Ag
Best material in fold 4:
Li0-3Sb
Best material in fold 5:
Li0-3Ce


## Testing on other DFT-bsaed features from materials project:
Testing to see if our model & scoring function can predict the quality of a battery based on untrained features, which still correlate with battery lifetime and hence the score.